# Home Credit 스파이크 2단계 — 다중 테이블 조인 피처 추가

1단계(`spike_home_credit_stage1.ipynb`)는 `application_train.csv` 단독으로 AUC 격차 +0.0126,
상위 5% 정밀도 격차 +3.1%p를 확인했다(GMC는 각각 +0.0093, +0.5%p). 이번 2단계는 보조 테이블
(`bureau`, `bureau_balance`, `previous_application`, `POS_CASH_balance`, `credit_card_balance`,
`installments_payments`)을 대출자(`SK_ID_CURR`) 단위로 집계해 피처로 추가한 뒤 같은 비교를 반복한다.

집계는 각 테이블에서 신용 위험과 직접 관련된 소수의 핵심 지표로 한정했다(완전 탐색이 아님 — Phase 5에서
더 정교하게 설계). 목적은 "다중 테이블 정보를 추가하면 격차가 더 벌어지는가"를 빠르게 확인하는 것이다.


In [1]:
import gc
import time
from contextlib import contextmanager

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
DATA_DIR = "../../data/raw_home_credit"
TARGET = "TARGET"

timings = {}


@contextmanager
def timer(step_name):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    timings[step_name] = elapsed
    print(f"[{step_name}] {elapsed:.1f}초")


## 1. 주 테이블 로드 (1단계와 동일)

In [2]:
with timer("주 테이블 로드"):
    train = pd.read_csv(f"{DATA_DIR}/application_train.csv", index_col="SK_ID_CURR")

print("shape:", train.shape)


[주 테이블 로드] 0.8초
shape: (307511, 121)


## 2. 보조 테이블 집계 (대출자 단위)

각 테이블을 `SK_ID_CURR` 단위로 집계해 원본 행 수와 무관하게 대출자 1명 = 1행으로 만든다.


In [3]:
with timer("bureau + bureau_balance 집계"):
    bureau = pd.read_csv(f"{DATA_DIR}/bureau.csv")
    bureau_balance = pd.read_csv(f"{DATA_DIR}/bureau_balance.csv")

    # bureau_balance: SK_ID_BUREAU 단위 연체 이력 -> SK_ID_BUREAU 단위로 먼저 요약
    bb_dpd = bureau_balance["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(int)
    bb_agg = bureau_balance.assign(DPD_FLAG=bb_dpd).groupby("SK_ID_BUREAU").agg(
        BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
        BB_DPD_RATIO=("DPD_FLAG", "mean"),
    )
    bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
    del bureau_balance, bb_agg
    gc.collect()

    bureau_agg = bureau.groupby("SK_ID_CURR").agg(
        BUREAU_COUNT=("SK_ID_BUREAU", "count"),
        BUREAU_ACTIVE_COUNT=("CREDIT_ACTIVE", lambda s: (s == "Active").sum()),
        BUREAU_CREDIT_SUM_TOTAL=("AMT_CREDIT_SUM", "sum"),
        BUREAU_CREDIT_SUM_DEBT_TOTAL=("AMT_CREDIT_SUM_DEBT", "sum"),
        BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
        BUREAU_BALANCE_DPD_RATIO_MEAN=("BB_DPD_RATIO", "mean"),
    )
    del bureau
    gc.collect()

print("bureau_agg shape:", bureau_agg.shape)
bureau_agg.head(3)


[bureau + bureau_balance 집계] 8.8초
bureau_agg shape: (305811, 7)


,BUREAU_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_CREDIT_SUM_TOTAL,BUREAU_CREDIT_SUM_DEBT_TOTAL,BUREAU_DAYS_CREDIT_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_BALANCE_DPD_RATIO_MEAN
SK_ID_CURR,,,,,,,
100001,7,3,1453365.000,596686.5,-735.00,0,0.007519
100002,8,2,865055.565,245781.0,-874.00,0,0.255682
100003,4,1,1017400.500,0.0,-1400.75,0,NaN


In [4]:
with timer("previous_application 집계"):
    prev = pd.read_csv(f"{DATA_DIR}/previous_application.csv")
    prev_agg = prev.groupby("SK_ID_CURR").agg(
        PREV_APP_COUNT=("SK_ID_PREV", "count"),
        PREV_APPROVED_RATIO=("NAME_CONTRACT_STATUS", lambda s: (s == "Approved").mean()),
        PREV_REFUSED_RATIO=("NAME_CONTRACT_STATUS", lambda s: (s == "Refused").mean()),
        PREV_AMT_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
        PREV_AMT_CREDIT_MEAN=("AMT_CREDIT", "mean"),
        PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
    )
    del prev
    gc.collect()

print("prev_agg shape:", prev_agg.shape)
prev_agg.head(3)


[previous_application 집계] 15.0초
prev_agg shape: (338857, 6)


,PREV_APP_COUNT,PREV_APPROVED_RATIO,PREV_REFUSED_RATIO,PREV_AMT_APPLICATION_MEAN,PREV_AMT_CREDIT_MEAN,PREV_DAYS_DECISION_MEAN
SK_ID_CURR,,,,,,
100001,1,1.0,0.0,24835.5,23787.0,-1740.0
100002,1,1.0,0.0,179055.0,179055.0,-606.0
100003,3,1.0,0.0,435436.5,484191.0,-1305.0


In [5]:
with timer("POS_CASH_balance 집계"):
    pos = pd.read_csv(f"{DATA_DIR}/POS_CASH_balance.csv")
    pos_agg = pos.groupby("SK_ID_CURR").agg(
        POS_COUNT=("SK_ID_PREV", "count"),
        POS_SK_DPD_MEAN=("SK_DPD", "mean"),
        POS_SK_DPD_MAX=("SK_DPD", "max"),
        POS_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    )
    del pos
    gc.collect()

print("pos_agg shape:", pos_agg.shape)
pos_agg.head(3)


[POS_CASH_balance 집계] 1.9초
pos_agg shape: (337252, 4)


,POS_COUNT,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MAX
SK_ID_CURR,,,,
100001,9,0.777778,7,7
100002,19,0.000000,0,0
100003,28,0.000000,0,0


In [6]:
with timer("credit_card_balance 집계"):
    cc = pd.read_csv(f"{DATA_DIR}/credit_card_balance.csv")
    cc["UTILIZATION"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
    cc_agg = cc.groupby("SK_ID_CURR").agg(
        CC_COUNT=("SK_ID_PREV", "count"),
        CC_AMT_BALANCE_MEAN=("AMT_BALANCE", "mean"),
        CC_AMT_BALANCE_MAX=("AMT_BALANCE", "max"),
        CC_SK_DPD_MAX=("SK_DPD", "max"),
        CC_UTILIZATION_MEAN=("UTILIZATION", "mean"),
    )
    del cc
    gc.collect()

print("cc_agg shape:", cc_agg.shape)
cc_agg.head(3)


[credit_card_balance 집계] 1.9초
cc_agg shape: (103558, 5)


,CC_COUNT,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_SK_DPD_MAX,CC_UTILIZATION_MEAN
SK_ID_CURR,,,,,
100006,6,0.000000,0.00,0,0.000000
100011,74,54482.111149,189000.00,0,0.302678
100013,96,18159.919219,161420.22,1,0.115301


In [7]:
with timer("installments_payments 집계"):
    inst = pd.read_csv(f"{DATA_DIR}/installments_payments.csv")
    inst["LATE_DAYS"] = (inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).clip(lower=0)
    inst["PAYMENT_RATIO"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
    inst_agg = inst.groupby("SK_ID_CURR").agg(
        INSTALL_COUNT=("SK_ID_PREV", "count"),
        INSTALL_LATE_DAYS_MEAN=("LATE_DAYS", "mean"),
        INSTALL_LATE_RATIO=("LATE_DAYS", lambda s: (s > 0).mean()),
        INSTALL_PAYMENT_RATIO_MEAN=("PAYMENT_RATIO", "mean"),
    )
    del inst
    gc.collect()

print("inst_agg shape:", inst_agg.shape)
inst_agg.head(3)


[installments_payments 집계] 7.8초
inst_agg shape: (339587, 4)


,INSTALL_COUNT,INSTALL_LATE_DAYS_MEAN,INSTALL_LATE_RATIO,INSTALL_PAYMENT_RATIO_MEAN
SK_ID_CURR,,,,
100001,7,1.571429,0.142857,1.0
100002,19,0.000000,0.000000,1.0
100003,25,0.000000,0.000000,1.0


## 3. 주 테이블에 조인

In [8]:
with timer("조인"):
    joined = train.join([bureau_agg, prev_agg, pos_agg, cc_agg, inst_agg], how="left")

new_feature_cols = (
    list(bureau_agg.columns) + list(prev_agg.columns) + list(pos_agg.columns)
    + list(cc_agg.columns) + list(inst_agg.columns)
)
print("조인 후 shape:", joined.shape, f"(신규 피처 {len(new_feature_cols)}개)")
print("\n신규 피처 결측률 (보조 테이블에 기록이 없는 대출자 = 결측):")
print(joined[new_feature_cols].isna().mean().sort_values(ascending=False))


[조인] 0.3초
조인 후 shape: (307511, 147) (신규 피처 26개)

신규 피처 결측률 (보조 테이블에 기록이 없는 대출자 = 결측):
CC_UTILIZATION_MEAN              0.720218
CC_SK_DPD_MAX                    0.717392
CC_AMT_BALANCE_MAX               0.717392
CC_AMT_BALANCE_MEAN              0.717392
CC_COUNT                         0.717392
BUREAU_BALANCE_DPD_RATIO_MEAN    0.700073
BUREAU_COUNT                     0.143149
BUREAU_CREDIT_SUM_TOTAL          0.143149
BUREAU_CREDIT_SUM_DEBT_TOTAL     0.143149
BUREAU_DAYS_CREDIT_MEAN          0.143149
BUREAU_CREDIT_DAY_OVERDUE_MAX    0.143149
BUREAU_ACTIVE_COUNT              0.143149
POS_SK_DPD_MAX                   0.058752
POS_SK_DPD_DEF_MAX               0.058752
POS_COUNT                        0.058752
POS_SK_DPD_MEAN                  0.058752
PREV_DAYS_DECISION_MEAN          0.053507
PREV_AMT_CREDIT_MEAN             0.053507
PREV_AMT_APPLICATION_MEAN        0.053507
PREV_REFUSED_RATIO               0.053507
PREV_APPROVED_RATIO              0.053507
PREV_APP_COUNT                  

## 4. 최소 전처리 (1단계와 동일 원칙 + 신규 피처)

In [9]:
with timer("전처리"):
    df = joined.copy()
    cat_cols = df.select_dtypes(include="object").columns.tolist()
    num_cols = [c for c in df.columns if c not in cat_cols + [TARGET]]

    df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[cat_cols] = df[cat_cols].fillna("Missing")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

    feature_cols = [c for c in df.columns if c != TARGET]
    lower = df[feature_cols].quantile(0.005)
    upper = df[feature_cols].quantile(0.995)
    df[feature_cols] = df[feature_cols].clip(lower=lower, upper=upper, axis=1)

print("최종 피처 수:", len(feature_cols))
print("결측 합계:", df[feature_cols].isna().sum().sum())


/var/folders/yk/y8jtmy9n3mqfnj05f__7_wqr0000gn/T/ipykernel_3293/1648386127.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


[전처리] 1.5초
최종 피처 수: 260
결측 합계: 0


## 5. 학습/홀드아웃 분할

In [10]:
X = df[feature_cols]
y = df[TARGET]
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print("학습:", X_train.shape, "홀드아웃:", X_holdout.shape)


학습: (246008, 260) 홀드아웃: (61503, 260)


## 6. 로지스틱 회귀 — 하이퍼파라미터 탐색

In [11]:
logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg_grid = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced"],
}

with timer("로지스틱 회귀 탐색"):
    logreg_search = GridSearchCV(logreg_pipe, logreg_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    logreg_search.fit(X_train, y_train)

logreg_best = logreg_search.best_estimator_
logreg_proba = logreg_best.predict_proba(X_holdout)[:, 1]
logreg_auc = roc_auc_score(y_holdout, logreg_proba)
print("최적 파라미터:", logreg_search.best_params_)
print("홀드아웃 AUC:", round(logreg_auc, 4))


[로지스틱 회귀 탐색] 62.3초
최적 파라미터: {'clf__C': 0.1, 'clf__class_weight': 'balanced'}
홀드아웃 AUC: 0.7636


## 7. XGBoost — 하이퍼파라미터 탐색

In [12]:
pos_weight_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_param_dist = {
    "n_estimators": randint(100, 400),
    "max_depth": randint(2, 8),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "scale_pos_weight": [1, 5, 10, round(pos_weight_ratio, 2)],
}

with timer("XGBoost 탐색"):
    xgb_search = RandomizedSearchCV(
        xgb.XGBClassifier(objective="binary:logistic", eval_metric="auc", random_state=RANDOM_STATE),
        param_distributions=xgb_param_dist,
        n_iter=25,
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_search.fit(X_train, y_train)

xgb_best = xgb_search.best_estimator_
xgb_proba = xgb_best.predict_proba(X_holdout)[:, 1]
xgb_auc = roc_auc_score(y_holdout, xgb_proba)
print("최적 파라미터:", xgb_search.best_params_)
print("홀드아웃 AUC:", round(xgb_auc, 4))


[XGBoost 탐색] 146.9초
최적 파라미터: {'colsample_bytree': np.float64(0.7301321323053057), 'learning_rate': np.float64(0.12271641400994977), 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 379, 'scale_pos_weight': 5, 'subsample': np.float64(0.9861021229056552)}
홀드아웃 AUC: 0.7767


## 8. AUC 비교 — 1단계·GMC 대비 격차 추이

In [13]:
GMC_LOGREG_AUC, GMC_XGB_AUC = 0.8598, 0.8691
HC1_LOGREG_AUC, HC1_XGB_AUC = 0.748669, 0.761305

comparison = pd.DataFrame(
    [
        ("GMC (변수 10개)", GMC_LOGREG_AUC, GMC_XGB_AUC),
        ("Home Credit 1단계 (주 테이블만)", HC1_LOGREG_AUC, HC1_XGB_AUC),
        ("Home Credit 2단계 (다중 테이블 조인)", logreg_auc, xgb_auc),
    ],
    columns=["데이터셋", "로지스틱 회귀 AUC", "XGBoost AUC"],
)
comparison["격차(XGB-LR)"] = comparison["XGBoost AUC"] - comparison["로지스틱 회귀 AUC"]
comparison


,데이터셋,로지스틱 회귀 AUC,XGBoost AUC,격차(XGB-LR)
0,GMC (변수 10개),0.859800,0.869100,0.009300
1,Home Credit 1단계 (주 테이블만),0.748669,0.761305,0.012636
2,Home Credit 2단계 (다중 테이블 조인),0.763595,0.776705,0.013109


## 9. 고위험 상위 5% 포착 성능 비교

In [14]:
def top_k_metrics(y_true, proba, k_ratio=0.05):
    n = len(y_true)
    k = int(np.ceil(n * k_ratio))
    order = np.argsort(-proba)
    top_idx = order[:k]
    y_true_arr = np.asarray(y_true)
    n_bad_in_top = y_true_arr[top_idx].sum()
    precision = n_bad_in_top / k
    recall = n_bad_in_top / y_true_arr.sum()
    lift = precision / y_true_arr.mean()
    return precision, recall, lift

logreg_p, logreg_r, logreg_l = top_k_metrics(y_holdout, logreg_proba)
xgb_p, xgb_r, xgb_l = top_k_metrics(y_holdout, xgb_proba)

HC1_LOGREG_P, HC1_LOGREG_R = 0.313069, 0.193958
HC1_XGB_P, HC1_XGB_R = 0.343953, 0.213092

top5_table = pd.DataFrame(
    [
        ("GMC", "로지스틱", 0.4747, 0.3551),
        ("GMC", "XGBoost", 0.4800, 0.3591),
        ("HC 1단계", "로지스틱", HC1_LOGREG_P, HC1_LOGREG_R),
        ("HC 1단계", "XGBoost", HC1_XGB_P, HC1_XGB_R),
        ("HC 2단계", "로지스틱", logreg_p, logreg_r),
        ("HC 2단계", "XGBoost", xgb_p, xgb_r),
    ],
    columns=["데이터셋", "모델", "정밀도(상위5%)", "포착률(상위5%)"],
)
top5_table


,데이터셋,모델,정밀도(상위5%),포착률(상위5%)
0,GMC,로지스틱,0.474700,0.355100
1,GMC,XGBoost,0.480000,0.359100
2,HC 1단계,로지스틱,0.313069,0.193958
3,HC 1단계,XGBoost,0.343953,0.213092
4,HC 2단계,로지스틱,0.331599,0.205438
5,HC 2단계,XGBoost,0.369961,0.229204


## 10. XGBoost 피처 중요도 상위 15개 — 새 조인 피처가 얼마나 쓰이는가

In [15]:
importance = pd.Series(xgb_best.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("신규 조인 피처 수:", len(new_feature_cols))
print("상위 15개 중 신규 조인 피처:", [f for f in importance.head(15).index if f in new_feature_cols])
importance.head(15)


신규 조인 피처 수: 26
상위 15개 중 신규 조인 피처: ['INSTALL_LATE_RATIO', 'PREV_REFUSED_RATIO', 'CC_UTILIZATION_MEAN']


EXT_SOURCE_2                                         0.077273
EXT_SOURCE_3                                         0.068143
NAME_EDUCATION_TYPE_Higher education                 0.046108
INSTALL_LATE_RATIO                                   0.027215
NAME_INCOME_TYPE_Working                             0.026268
CODE_GENDER_M                                        0.023584
FLAG_DOCUMENT_3                                      0.021936
PREV_REFUSED_RATIO                                   0.020364
DAYS_EMPLOYED                                        0.019293
CC_UTILIZATION_MEAN                                  0.018823
EXT_SOURCE_1                                         0.017845
OWN_CAR_AGE                                          0.015794
ELEVATORS_MEDI                                       0.014913
NAME_EDUCATION_TYPE_Secondary / secondary special    0.014571
OCCUPATION_TYPE_Sales staff                          0.014238
dtype: float32

In [16]:
timing_df = pd.DataFrame([(k, f"{v:.1f}초") for k, v in timings.items()], columns=["단계", "소요 시간"])
print(f"전체 합계: {sum(timings.values()):.1f}초")
timing_df


전체 합계: 247.3초


,단계,소요 시간
0,주 테이블 로드,0.8초
1,bureau + bureau_balance 집계,8.8초
2,previous_application 집계,15.0초
3,POS_CASH_balance 집계,1.9초
4,credit_card_balance 집계,1.9초
5,installments_payments 집계,7.8초
6,조인,0.3초
7,전처리,1.5초
8,로지스틱 회귀 탐색,62.3초
9,XGBoost 탐색,146.9초


## 11. 결론

- 8절·9절에서 격차가 1단계보다 더 벌어졌다면 → "다중 테이블 조인만큼 비선형 모델 우위가 커진다"는 근거 추가.
- 10절에서 신규 조인 피처가 상위권에 많이 등장했다면 → 다중 테이블 피처 엔지니어링이 실제로 유효했다는 뜻이고,
  동시에 SHAP 설명에서 "왜 이 조인 피처가 위험 신호인지"를 사유 코드로 자연스럽게 풀어쓸 수 있는지가 Phase 7의 과제가 된다.
- 결과는 `docs/spike-home-credit.md`에 반영한다.
